# Demo — Metadata as a Governance Control (live AWS Glue)

**Trailhead Provisions** runs as a federation of four domain teams on a shared **AWS Glue Data
Catalog**. This short walkthrough demonstrates the **technique** you'll use in the exercise:
treat the catalog as a *control surface* — connect to live AWS Glue, run the audit, read a finding,
and **fix it in the catalog**, then re-audit to confirm. Here we remediate two gaps in the
E-commerce Orders and Inventory & Fulfillment domains; the exercise asks you to do the full
audit-and-remediate for the Customer and Marketing domains yourself.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

created Glue database 'trailhead'
registered s3://123456789012-trailhead-data with Lake Formation
  customer               rows=  310  table-gaps=[]  untagged-cols=['email']
  orders                 rows= 1200  table-gaps=['owner']  untagged-cols=[]
  order_returns          rows=  106  table-gaps=[]  untagged-cols=[]
  supplier               rows=   40  table-gaps=['retention']  untagged-cols=[]
  product                rows=  200  table-gaps=['retention']  untagged-cols=['unit_cost']
  shipment               rows= 1109  table-gaps=['classification']  untagged-cols=[]
  campaign               rows=   20  table-gaps=['owner']  untagged-cols=[]
  campaign_membership    rows=  800  table-gaps=['owner', 'retention']  untagged-cols=['cust_ref']

Done. Glue database 'trailhead' seeded in s3://123456789012-trailhead-data with Lake Formation enforcement enabled.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)


## Run the catalog audit
One row per governance gap, read from the Glue catalog across all four domains.

In [3]:
audit = gc.catalog_audit()
print('Total gaps:', len(audit))
audit

Total gaps: 10


,table,domain,gap_type,detail
0,customer,Customer Identity & Loyalty,untagged_column,email (type TEXT)
1,orders,E-commerce Orders,missing_owner,owner
2,supplier,Inventory & Fulfillment,missing_retention,retention
3,product,Inventory & Fulfillment,missing_retention,retention
4,product,Inventory & Fulfillment,untagged_column,unit_cost (type REAL)
5,shipment,Inventory & Fulfillment,missing_classification,classification
6,campaign,Marketing & Campaigns,missing_owner,owner
7,campaign_membership,Marketing & Campaigns,missing_owner,owner
8,campaign_membership,Marketing & Campaigns,missing_retention,retention
9,campaign_membership,Marketing & Campaigns,untagged_column,cust_ref (type TEXT)


## How to read the audit (two representative findings)

**A table-level gap — `orders` has no owner.** Filter to one table to see its gaps:

In [4]:
audit[audit["table"] == "orders"]

,table,domain,gap_type,detail
1,orders,E-commerce Orders,missing_owner,owner


A missing `owner` means no one is accountable for purchase events. **Fix it in the
catalog** with `set_table_metadata`, which writes the Glue table parameters:

In [5]:
gc.set_table_metadata("orders", owner="orders-team@trailhead.example")
gc.catalog_audit().pipe(lambda d: d[d["table"] == "orders"])   # gap for orders is gone

,table,domain,gap_type,detail


**A column-level gap — an untagged column.** Columns with no classification are invisible to
tag-driven access controls. Filter to see every `untagged_column` gap across the catalog:

In [6]:
audit[audit["gap_type"] == "untagged_column"]

,table,domain,gap_type,detail
0,customer,Customer Identity & Loyalty,untagged_column,email (type TEXT)
4,product,Inventory & Fulfillment,untagged_column,unit_cost (type REAL)
9,campaign_membership,Marketing & Campaigns,untagged_column,cust_ref (type TEXT)


In the Inventory & Fulfillment domain, `product.unit_cost` holds commercially sensitive wholesale
pricing but has no classification tag. Tag it with `tag_column`, which writes the column's Glue
classification parameter (leaving the Customer and Marketing PII gaps for you to remediate in the
exercise):

In [7]:
gc.tag_column("product", "unit_cost", "SENSITIVE")
gc.catalog_audit().pipe(lambda d: d[d["gap_type"] == "untagged_column"])

,table,domain,gap_type,detail
0,customer,Customer Identity & Loyalty,untagged_column,email (type TEXT)
7,campaign_membership,Marketing & Campaigns,untagged_column,cust_ref (type TEXT)


## Gaps aren't only missing tags — domains also disagree
The audit's companion is the semantic-conflict report: places where domains define the same concept differently.

In [8]:
gc.semantic_conflicts()

,concept,domain_a,field_a,domain_b,field_b,impact
0,customer identity key,Customer Identity & Loyalty,customer.customer_id (INTEGER surrogate),Marketing & Campaigns,campaign_membership.cust_ref (email string),No shared join key; cross-domain customer reso...
1,marketing consent,Customer Identity & Loyalty,customer.marketing_consent,Marketing & Campaigns,campaign_membership.mkt_consent,Two systems of record for consent disagree ~20...
2,carbon unit of measure,Inventory & Fulfillment,shipment.carbon_kg (kg),Reporting (derived),a slice stored in grams,Mixed units inflate the CSRD disclosure unless...


Read the **consent** conflict: loyalty and marketing each keep their own consent
flag, and they disagree — acting on the wrong copy is a GDPR violation, surfaced entirely
through metadata.

**Takeaway / your turn:** the catalog is a control surface — audit it, **remediate the gaps in
Glue**, and re-audit to verify. In the exercise you'll do this for the Customer and Marketing
domains (including their untagged `PII` columns) and write the reconciliation memo.